# face_qef GPU Compare

JIT compile current `intersect_qef`, `face_qef_old`, `face_qef_ref`, current `face_qef`, and a CPU `face_qef` wrapper. `intersect_qef` only prepares the active voxel and brick inputs used by the face QEF variants.

In [1]:
import os
import sys
import time
import tempfile
import statistics
from pathlib import Path

CONDA_PREFIX = Path(os.environ.get('CONDA_PREFIX', '/home/quanta/.conda/envs/symm-enforce'))
os.environ['PATH'] = f"{CONDA_PREFIX / 'bin'}:{os.environ.get('PATH', '')}"
os.environ['CUDA_HOME'] = str(CONDA_PREFIX)
os.environ['CC'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['CXX'] = str(CONDA_PREFIX / 'bin/g++')
os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)

import torch
import torch.utils.cpp_extension as cpp_extension
from torch.utils.cpp_extension import load

cpp_extension.CUDA_HOME = str(CONDA_PREFIX)
torch.set_grad_enabled(False)

ROOT = Path.cwd()
if not (ROOT / 'setup.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'setup.py').exists(), f'Cannot find repo root from {Path.cwd()}'

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

GRID_SIZE = int(os.environ.get('FDG_GRID_SIZE', '512'))
CHUNK_TRIANGLES = int(os.environ.get('FDG_CHUNK_TRIANGLES', '20000'))
SUBDIVIDE_AREA_THRESHOLD = float(os.environ.get('FDG_SUBDIVIDE_AREA_THRESHOLD', str(1.0 / (GRID_SIZE * GRID_SIZE))))
SUBDIVIDE_MAX_ITERS = int(os.environ.get('FDG_SUBDIVIDE_MAX_ITERS', '12'))
WARM_REPEATS = int(os.environ.get('FDG_WARM_REPEATS', '10'))
RUN_CPU_SUBDIVIDED = os.environ.get('FDG_RUN_CPU_SUBDIVIDED', '1') != '0'

print('repo:', ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('CONDA_PREFIX:', CONDA_PREFIX)
print('nvcc:', CONDA_PREFIX / 'bin/nvcc')
print('gcc:', os.environ['CC'])
print('g++:', os.environ['CXX'])
print('MAX_JOBS:', os.environ['MAX_JOBS'])
print('GRID_SIZE:', GRID_SIZE, 'CHUNK_TRIANGLES:', CHUNK_TRIANGLES)
print('SUBDIVIDE_AREA_THRESHOLD:', SUBDIVIDE_AREA_THRESHOLD, 'SUBDIVIDE_MAX_ITERS:', SUBDIVIDE_MAX_ITERS)
print('WARM_REPEATS:', WARM_REPEATS, 'RUN_CPU_SUBDIVIDED:', RUN_CPU_SUBDIVIDED)


repo: /mnt/nvmefs/Projects/o-voxel-gpu
python: /home/quanta/.conda/envs/symm-enforce/bin/python
torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
CONDA_PREFIX: /home/quanta/.conda/envs/symm-enforce
nvcc: /home/quanta/.conda/envs/symm-enforce/bin/nvcc
gcc: /home/quanta/.conda/envs/symm-enforce/bin/gcc
g++: /home/quanta/.conda/envs/symm-enforce/bin/g++
MAX_JOBS: 32
GRID_SIZE: 512 CHUNK_TRIANGLES: 20000
SUBDIVIDE_AREA_THRESHOLD: 3.814697265625e-06 SUBDIVIDE_MAX_ITERS: 12
WARM_REPEATS: 10 RUN_CPU_SUBDIVIDED: True


## JIT Extension

The temporary C++ binding includes the CPU `face_qef` implementation from the current CPU source and binds the three GPU face QEF variants from the CUDA module.

In [2]:
build_dir = Path(tempfile.mkdtemp(prefix='fdg_face_qef_jit_'))
binding_path = build_dir / 'binding.cpp'
torch_build_dir = build_dir / 'torch_ext'
torch_build_dir.mkdir(parents=True, exist_ok=True)

binding_cpp = r"""
#include <torch/extension.h>
#include <Eigen/Dense>
#include <unordered_map>
#include <vector>
#include "__ROOT__/src/convert/flexible_dual_grid.cpp"
#include "__ROOT__/src/convert/mesh_to_flexible_dual_grid_gpu/api.h"

namespace py = pybind11;

namespace {

torch::Tensor face_qef_cpu_only(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxels) {
    const int64_t T = triangles.size(0);
    const int64_t N = voxels.size(0);
    const float* tri_ptr = triangles.data_ptr<float>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vox_ptr = voxels.data_ptr<int32_t>();

    std::vector<Eigen::Vector3f> tri_vec;
    tri_vec.reserve(static_cast<size_t>(T) * 3);
    for (int64_t t = 0; t < T; ++t) {
        tri_vec.emplace_back(tri_ptr[9 * t + 0], tri_ptr[9 * t + 1], tri_ptr[9 * t + 2]);
        tri_vec.emplace_back(tri_ptr[9 * t + 3], tri_ptr[9 * t + 4], tri_ptr[9 * t + 5]);
        tri_vec.emplace_back(tri_ptr[9 * t + 6], tri_ptr[9 * t + 7], tri_ptr[9 * t + 8]);
    }

    std::unordered_map<VoxelCoord, size_t> hash_table;
    hash_table.reserve(static_cast<size_t>(N) * 2);
    for (int64_t i = 0; i < N; ++i) {
        hash_table[VoxelCoord{vox_ptr[3 * i + 0], vox_ptr[3 * i + 1], vox_ptr[3 * i + 2]}] = static_cast<size_t>(i);
    }

    std::vector<Eigen::Matrix4f> qefs(static_cast<size_t>(N), Eigen::Matrix4f::Zero());
    ::face_qef(
        Eigen::Vector3f(vs[0], vs[1], vs[2]),
        Eigen::Vector3i(gr[0], gr[1], gr[2]),
        Eigen::Vector3i(gr[3], gr[4], gr[5]),
        tri_vec,
        hash_table,
        qefs);

    auto out = torch::empty({N, 10}, torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU));
    float* out_ptr = out.data_ptr<float>();
    for (int64_t i = 0; i < N; ++i) {
        const Eigen::Matrix4f& Q = qefs[static_cast<size_t>(i)];
        out_ptr[10 * i + 0] = Q(0, 0);
        out_ptr[10 * i + 1] = Q(0, 1);
        out_ptr[10 * i + 2] = Q(0, 2);
        out_ptr[10 * i + 3] = Q(0, 3);
        out_ptr[10 * i + 4] = Q(1, 1);
        out_ptr[10 * i + 5] = Q(1, 2);
        out_ptr[10 * i + 6] = Q(1, 3);
        out_ptr[10 * i + 7] = Q(2, 2);
        out_ptr[10 * i + 8] = Q(2, 3);
        out_ptr[10 * i + 9] = Q(3, 3);
    }
    return out;
}

} // namespace

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("intersect_qef", &o_voxel::fdg::intersect_qef);
    m.def("face_qef_old", &o_voxel::fdg::face_qef_old);
    m.def("face_qef_ref", &o_voxel::fdg::face_qef_ref);
    m.def("face_qef", &o_voxel::fdg::face_qef);
    m.def("face_qef_cpu", &face_qef_cpu_only);
}
""".replace('__ROOT__', ROOT.as_posix())

binding_path.write_text(binding_cpp)

sources = [
    str(binding_path),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_octree.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_old.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_ref.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef.cu'),
]

ext = load(
    name=f'fdg_face_qef_jit_{os.getpid()}',
    sources=sources,
    build_directory=str(torch_build_dir),
    extra_include_paths=[
        str(ROOT / 'src/convert'),
        str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu'),
        str(ROOT / 'third_party/eigen'),
    ],
    extra_cflags=['-O3', '-std=c++17'],
    extra_cuda_cflags=['-O3', '-std=c++17'],
    with_cuda=True,
    verbose=True,
)
print('JIT extension loaded:', ext)


Detected CUDA files, patching ldflags
Emitting ninja build file /tmp/fdg_face_qef_jit_r878j1n5/torch_ext/build.ninja...
/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fdg_face_qef_jit_866718...
Using envvar MAX_JOBS (32) as the number of workers...


[1/7] /home/quanta/.conda/envs/symm-enforce/bin/g++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fdg_face_qef_jit_866718 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert/mesh_to_flexible_dual_grid_gpu -I/mnt/nvmefs/Projects/o-voxel-gpu/third_party/eigen -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -O3 -std=c+

Loading extension module fdg_face_qef_jit_866718...


## Load Mesh Inputs

The original mesh comes from `notebooks/test.glb`. The subdivided input uses the same GPU subdivision helper as the intersect notebook.

In [3]:
import numpy as np
import trimesh


def subdivide_mesh_gpu(verts: torch.Tensor, faces: torch.Tensor, area_threshold: float, max_iters: int):
    from pytorch3d.structures import Meshes

    device = verts.device
    cur = Meshes(verts=[verts], faces=[faces])

    for _ in range(max_iters):
        cv = cur.verts_packed()
        cf = cur.faces_packed()
        edges = cur.edges_packed()
        f2e = cur.faces_packed_to_edges_packed()

        areas = cur.faces_areas_packed()
        large = areas > area_threshold
        if not large.any():
            break

        edge_len = torch.norm(cv[edges[:, 0]] - cv[edges[:, 1]], dim=1)
        _, local_max = torch.max(edge_len[f2e], dim=1)
        max_edge_ids = f2e[torch.arange(cf.shape[0], device=device), local_max]

        target = torch.zeros(edges.shape[0], dtype=torch.bool, device=device)
        target[max_edge_ids[large]] = True
        if not target.any():
            break

        mid = (cv[edges[target, 0]] + cv[edges[target, 1]]) * 0.5
        new_verts = torch.cat([cv, mid], dim=0)
        n_old = cv.shape[0]

        e2new = torch.zeros(edges.shape[0], dtype=torch.long, device=device)
        e2new[target] = torch.arange(n_old, n_old + target.sum().item(), device=device)

        split_counts = target[f2e].sum(dim=1)

        v0, v1, v2 = cf[:, 0], cf[:, 1], cf[:, 2]
        p01_a = torch.minimum(v0, v1)
        p01_b = torch.maximum(v0, v1)
        p12_a = torch.minimum(v1, v2)
        p12_b = torch.maximum(v1, v2)
        p20_a = torch.minimum(v2, v0)
        p20_b = torch.maximum(v2, v0)

        e_act = edges[f2e]
        match01 = (e_act[:, :, 0] == p01_a.unsqueeze(1)) & (e_act[:, :, 1] == p01_b.unsqueeze(1))
        match12 = (e_act[:, :, 0] == p12_a.unsqueeze(1)) & (e_act[:, :, 1] == p12_b.unsqueeze(1))
        match20 = (e_act[:, :, 0] == p20_a.unsqueeze(1)) & (e_act[:, :, 1] == p20_b.unsqueeze(1))

        col01 = match01.long().argmax(dim=1)
        col12 = match12.long().argmax(dim=1)
        col20 = match20.long().argmax(dim=1)

        keep = cf[split_counts == 0]
        out = [keep]

        m1 = split_counts == 1
        if m1.any():
            f1 = cf[m1]
            fe1 = f2e[m1]
            M = f1.shape[0]
            v0_1, v1_1, v2_1 = f1[:, 0], f1[:, 1], f1[:, 2]

            split_col = target[fe1].long().argmax(dim=1)
            c01_m1 = col01[m1]
            c12_m1 = col12[m1]
            c20_m1 = col20[m1]
            is_split_01 = split_col == c01_m1
            is_split_12 = split_col == c12_m1
            is_split_20 = split_col == c20_m1

            split_eid = fe1[torch.arange(M, device=device), split_col]
            vn = e2new[split_eid]

            s1 = torch.zeros(M, 3, dtype=torch.long, device=device)
            s2 = torch.zeros(M, 3, dtype=torch.long, device=device)

            s1[is_split_01] = torch.stack([v0_1[is_split_01], vn[is_split_01], v2_1[is_split_01]], dim=1)
            s2[is_split_01] = torch.stack([vn[is_split_01], v1_1[is_split_01], v2_1[is_split_01]], dim=1)
            s1[is_split_12] = torch.stack([v0_1[is_split_12], v1_1[is_split_12], vn[is_split_12]], dim=1)
            s2[is_split_12] = torch.stack([v0_1[is_split_12], vn[is_split_12], v2_1[is_split_12]], dim=1)
            s1[is_split_20] = torch.stack([v0_1[is_split_20], v1_1[is_split_20], vn[is_split_20]], dim=1)
            s2[is_split_20] = torch.stack([vn[is_split_20], v1_1[is_split_20], v2_1[is_split_20]], dim=1)

            out.extend([s1, s2])

        mm = split_counts >= 2
        if mm.any():
            fm = cf[mm]
            fem = f2e[mm]
            M2 = fm.shape[0]
            v0_m, v1_m, v2_m = fm[:, 0], fm[:, 1], fm[:, 2]

            c01_m = col01[mm]
            c12_m = col12[mm]
            c20_m = col20[mm]

            eid01 = fem[torch.arange(M2, device=device), c01_m]
            e01_is_split = target[eid01]
            e01_v = torch.where(e01_is_split, e2new[eid01], v0_m)

            eid12 = fem[torch.arange(M2, device=device), c12_m]
            e12_is_split = target[eid12]
            e12_v = torch.where(e12_is_split, e2new[eid12], v1_m)

            eid20 = fem[torch.arange(M2, device=device), c20_m]
            e20_is_split = target[eid20]
            e20_v = torch.where(e20_is_split, e2new[eid20], v2_m)

            mf1 = torch.stack([v0_m, e01_v, e20_v], dim=1)
            mf2 = torch.stack([e01_v, v1_m, e12_v], dim=1)
            mf3 = torch.stack([e20_v, e12_v, v2_m], dim=1)
            mf4 = torch.stack([e01_v, e12_v, e20_v], dim=1)
            out.extend([mf1, mf2, mf3, mf4])

        cur = Meshes(verts=[new_verts], faces=[torch.cat(out, dim=0)])

    return cur.verts_packed(), cur.faces_packed()


def triangle_mb(triangles: torch.Tensor) -> float:
    return triangles.numel() * triangles.element_size() / 1024**2


glb_path = ROOT / 'notebooks/test.glb'
loaded = trimesh.load(glb_path, force='scene')
if isinstance(loaded, trimesh.Scene):
    meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    mesh = trimesh.util.concatenate(meshes)
else:
    mesh = loaded
assert isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0

vertices_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)
vmin = vertices_np.min(axis=0)
vmax = vertices_np.max(axis=0)
center = (vmin + vmax) * 0.5
scale = np.float32(0.999 / max((vmax - vmin).max(), 1e-12))
vertices_np = (vertices_np - center) * scale + np.float32(0.5)

vertices_cpu = torch.from_numpy(vertices_np).contiguous()
faces_cpu = torch.from_numpy(faces_np).contiguous()
triangles_cpu = vertices_cpu[faces_cpu].reshape(-1, 9).contiguous()
triangles_cuda = triangles_cpu.to(DEVICE, non_blocking=True).contiguous()

voxel_size_cpu = torch.tensor([1.0 / GRID_SIZE, 1.0 / GRID_SIZE, 1.0 / GRID_SIZE], dtype=torch.float32)
grid_range_cpu = torch.tensor([0, 0, 0, GRID_SIZE, GRID_SIZE, GRID_SIZE], dtype=torch.int32)

print('building subdivided input...')
torch.cuda.empty_cache()
torch.cuda.synchronize()
start = time.perf_counter()
sub_vertices_cuda, sub_faces_cuda = subdivide_mesh_gpu(
    vertices_cpu.to(DEVICE, non_blocking=True).contiguous(),
    faces_cpu.to(DEVICE, non_blocking=True).contiguous(),
    area_threshold=SUBDIVIDE_AREA_THRESHOLD,
    max_iters=SUBDIVIDE_MAX_ITERS,
)
sub_triangles_cuda = sub_vertices_cuda[sub_faces_cuda].reshape(-1, 9).contiguous()
torch.cuda.synchronize()
subdivide_ms = (time.perf_counter() - start) * 1000.0
sub_triangles_cpu = sub_triangles_cuda.detach().cpu().contiguous()
sub_vertices_shape = tuple(sub_vertices_cuda.shape)
sub_faces_shape = tuple(sub_faces_cuda.shape)
del sub_vertices_cuda, sub_faces_cuda
torch.cuda.empty_cache()

INPUT_CASES = {
    'original': {
        'triangles_cpu': triangles_cpu,
        'triangles_cuda': triangles_cuda,
        'vertices_shape': tuple(vertices_cpu.shape),
        'faces_shape': tuple(faces_cpu.shape),
        'triangles': int(triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(triangles_cuda),
        'preprocess_ms': 0.0,
    },
    'subdivided': {
        'triangles_cpu': sub_triangles_cpu,
        'triangles_cuda': sub_triangles_cuda,
        'vertices_shape': sub_vertices_shape,
        'faces_shape': sub_faces_shape,
        'triangles': int(sub_triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(sub_triangles_cuda),
        'preprocess_ms': subdivide_ms,
    },
}

for name, case in INPUT_CASES.items():
    print(
        name,
        'vertices:', case['vertices_shape'],
        'faces:', case['faces_shape'],
        'triangles:', case['triangles'],
        'triangle_mb:', case['triangle_mb'],
        'preprocess_ms:', case['preprocess_ms'],
    )
print('face multiplier:', float(INPUT_CASES['subdivided']['triangles'] / max(INPUT_CASES['original']['triangles'], 1)))
print('voxel_size:', voxel_size_cpu.tolist(), 'grid_range:', grid_range_cpu.tolist())


building subdivided input...
original vertices: (268018, 3) faces: (280333, 3) triangles: 280333 triangle_mb: 9.624469757080078 preprocess_ms: 0.0
subdivided vertices: (3608864, 3) faces: (10304077, 3) triangles: 10304077 triangle_mb: 353.7624092102051 preprocess_ms: 798.0357529995672
face multiplier: 36.75656094715927
voxel_size: [0.001953125, 0.001953125, 0.001953125] grid_range: [0, 0, 0, 512, 512, 512]


## Helpers

GPU functions take CUDA triangles. `voxel_size` and `grid_range` stay as CPU tensors because the C++ functions read their scalar values on host.

In [4]:
from typing import NamedTuple


class IntersectOutput(NamedTuple):
    voxels: torch.Tensor
    mean_sum: torch.Tensor
    cnt: torch.Tensor
    intersected: torch.Tensor
    qefs: torch.Tensor
    brick_hash_keys: torch.Tensor
    brick_hash_vals: torch.Tensor
    brick_bits: torch.Tensor
    brick_base: torch.Tensor


def as_intersect_output(output):
    assert len(output) == 9, f'expected 9 tensors from intersect_qef, got {len(output)}'
    return IntersectOutput(*output)


def voxel_key(voxels: torch.Tensor) -> torch.Tensor:
    voxels = voxels.to(torch.int64)
    rel = voxels - grid_range_cpu[:3].to(torch.int64)
    size = (grid_range_cpu[3:] - grid_range_cpu[:3]).to(torch.int64)
    return rel[:, 0] + size[0] * (rel[:, 1] + size[1] * rel[:, 2])


def canonical_face_qef(voxels: torch.Tensor, qefs: torch.Tensor):
    voxels_cpu = voxels.detach().cpu().contiguous()
    qefs_cpu = qefs.detach().cpu().float().contiguous()
    keys = voxel_key(voxels_cpu)
    order = torch.argsort(keys)
    keys = keys[order]
    unique = torch.unique_consecutive(keys).numel() == keys.numel()
    return {
        'keys': keys,
        'voxels': voxels_cpu[order],
        'qefs': qefs_cpu[order],
        'unique': bool(unique),
    }


def compare_face_qef(cpu, other):
    row = {
        'count_cpu': int(cpu['keys'].numel()),
        'count_other': int(other['keys'].numel()),
        'unique_other': other['unique'],
    }
    if not torch.equal(cpu['keys'], other['keys']):
        a = set(cpu['keys'].tolist())
        b = set(other['keys'].tolist())
        row.update({
            'voxel_keys_equal': False,
            'missing': len(a - b),
            'extra': len(b - a),
            'missing_sample': sorted(list(a - b))[:8],
            'extra_sample': sorted(list(b - a))[:8],
        })
        return row
    diff = (other['qefs'] - cpu['qefs']).float()
    ref = cpu['qefs'].float()
    row.update({
        'voxel_keys_equal': True,
        'missing': 0,
        'extra': 0,
        'qef_max_abs': float(diff.abs().max().item()) if diff.numel() else 0.0,
        'qef_rmse': float(torch.sqrt(torch.mean(diff * diff)).item()) if diff.numel() else 0.0,
        'qef_rel_l2': float(torch.linalg.vector_norm(diff).item() / max(torch.linalg.vector_norm(ref).item(), 1e-12)) if diff.numel() else 0.0,
        'qef_cpu_max_abs': float(ref.abs().max().item()) if ref.numel() else 0.0,
        'qef_other_max_abs': float(other['qefs'].abs().max().item()) if other['qefs'].numel() else 0.0,
    })
    return row


def show_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)


def face_old(case, inter):
    return ext.face_qef_old(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, inter.voxels)


def face_ref(case, inter):
    return ext.face_qef_ref(
        case['triangles_cuda'], voxel_size_cpu, grid_range_cpu,
        inter.voxels, inter.brick_hash_keys, inter.brick_hash_vals, inter.brick_bits, inter.brick_base)


def face_new(case, inter):
    return ext.face_qef(
        case['triangles_cuda'], voxel_size_cpu, grid_range_cpu,
        inter.voxels, inter.brick_hash_keys, inter.brick_hash_vals, inter.brick_bits, inter.brick_base)


def face_cpu(case, inter):
    return ext.face_qef_cpu(case['triangles_cpu'], voxel_size_cpu, grid_range_cpu, inter.voxels.detach().cpu().contiguous())


## Build Active Voxel Inputs

This cell runs only the current `intersect_qef`. Its output supplies the shared `voxels` and brick lookup tensors for all face QEF measurements.

In [5]:
torch.cuda.empty_cache()
torch.cuda.synchronize()

INTERSECT_OUTPUTS = {}
intersect_rows = []
for input_name, case in INPUT_CASES.items():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = as_intersect_output(ext.intersect_qef(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES))
    end.record()
    torch.cuda.synchronize()
    INTERSECT_OUTPUTS[input_name] = out
    intersect_rows.append({
        'input': input_name,
        'triangles': case['triangles'],
        'voxels': int(out.voxels.shape[0]),
        'brick_hash_capacity': int(out.brick_hash_keys.shape[0]),
        'brick_slots': int(out.brick_bits.shape[0]),
        'intersect_ms': float(start.elapsed_time(end)),
        'intersect_peak_alloc_mb': float((torch.cuda.max_memory_allocated() - base_alloc) / 1024**2),
        'intersect_peak_reserved_mb': float((torch.cuda.max_memory_reserved() - base_reserved) / 1024**2),
    })

show_rows(intersect_rows)


,input,triangles,voxels,brick_hash_capacity,brick_slots,intersect_ms,intersect_peak_alloc_mb,intersect_peak_reserved_mb
0,original,280333,4676307,524288,262144,27.489376,382.233398,340.0
1,subdivided,10304077,4676306,524288,262144,47.897598,1227.611816,1202.0


## Correctness Against CPU face_qef

All outputs are canonicalized by voxel key before error metrics, so row-order differences do not affect evaluation.

In [6]:
FACE_CANON = {}
correctness_rows = []

for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    cpu_allowed = input_name != 'subdivided' or RUN_CPU_SUBDIVIDED
    if not cpu_allowed:
        print(f'skip CPU face_qef for {input_name}; set FDG_RUN_CPU_SUBDIVIDED=1 to enable')
        continue

    print('running CPU face_qef for', input_name, 'triangles:', case['triangles'], 'voxels:', int(inter.voxels.shape[0]))
    cpu_qef = face_cpu(case, inter)
    cpu_can = canonical_face_qef(inter.voxels, cpu_qef)
    FACE_CANON[(input_name, 'cpu')] = cpu_can
    assert cpu_can['unique'], f'{input_name} CPU voxel keys are not unique'

    gpu_outputs = {
        'old': face_old(case, inter),
        'ref': face_ref(case, inter),
        'new': face_new(case, inter),
    }
    torch.cuda.synchronize()

    for method, qef in gpu_outputs.items():
        can = canonical_face_qef(inter.voxels, qef)
        FACE_CANON[(input_name, method)] = can
        row = {'input': input_name, 'method': method, **compare_face_qef(cpu_can, can)}
        correctness_rows.append(row)
        assert row['voxel_keys_equal'], f'{input_name} {method} voxel keys differ from CPU'

show_rows(correctness_rows)


running CPU face_qef for original triangles: 280333 voxels: 4676307
running CPU face_qef for subdivided triangles: 10304077 voxels: 4676306


,input,method,count_cpu,count_other,unique_other,voxel_keys_equal,missing,extra,qef_max_abs,qef_rmse,qef_rel_l2,qef_cpu_max_abs,qef_other_max_abs
0,original,old,4676307,4676307,True,True,0,0,4.360647,0.003829,0.004826,52.151131,52.151131
1,original,ref,4676307,4676307,True,True,0,0,1.903503,0.003650,0.004600,52.151131,52.151127
2,original,new,4676307,4676307,True,True,0,0,1.903502,0.003652,0.004603,52.151131,52.151131
3,subdivided,old,4676306,4676306,True,True,0,0,74.999954,0.637490,0.217004,98.073151,160.740829
4,subdivided,ref,4676306,4676306,True,True,0,0,74.067192,0.634783,0.216085,98.073151,160.737869
5,subdivided,new,4676306,4676306,True,True,0,0,74.067207,0.634782,0.216085,98.073151,160.737854


## GPU Timing And Memory

Cold runs clear the caching allocator before the call. Warm results report the median across repeats. Memory numbers are peak deltas relative to the live inputs from `intersect_qef`.

In [7]:
def timed_cuda_call(fn, cold=False):
    if cold:
        torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = fn()
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    peak_alloc = torch.cuda.max_memory_allocated() - base_alloc
    peak_reserved = torch.cuda.max_memory_reserved() - base_reserved
    del out
    torch.cuda.synchronize()
    return {
        'ms': float(ms),
        'peak_alloc_mb': float(peak_alloc / 1024**2),
        'peak_reserved_mb': float(peak_reserved / 1024**2),
    }


def summarize_records(records):
    return {
        'ms_median': statistics.median(r['ms'] for r in records),
        'ms_min': min(r['ms'] for r in records),
        'peak_alloc_mb_median': statistics.median(r['peak_alloc_mb'] for r in records),
        'peak_reserved_mb_median': statistics.median(r['peak_reserved_mb'] for r in records),
    }


bench_rows = []
for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    methods = {
        'old': lambda case=case, inter=inter: face_old(case, inter),
        'ref': lambda case=case, inter=inter: face_ref(case, inter),
        'new': lambda case=case, inter=inter: face_new(case, inter),
    }
    for method, fn in methods.items():
        cold = timed_cuda_call(fn, cold=True)
        _ = timed_cuda_call(fn, cold=False)
        warm_records = [timed_cuda_call(fn, cold=False) for _ in range(WARM_REPEATS)]
        warm = summarize_records(warm_records)
        bench_rows.append({
            'input': input_name,
            'triangles': case['triangles'],
            'voxels': int(inter.voxels.shape[0]),
            'input_triangle_mb': case['triangle_mb'],
            'method': method,
            'cold_ms': cold['ms'],
            'cold_peak_alloc_mb': cold['peak_alloc_mb'],
            'cold_peak_reserved_mb': cold['peak_reserved_mb'],
            **warm,
        })

show_rows(bench_rows)


,input,triangles,voxels,input_triangle_mb,method,cold_ms,cold_peak_alloc_mb,cold_peak_reserved_mb,ms_median,ms_min,peak_alloc_mb_median,peak_reserved_mb_median
0,original,280333,4676307,9.624470,old,33.359009,1552.552246,1108.0,27.527761,26.460159,1553.522461,0.0
1,original,280333,4676307,9.624470,ref,30.987265,178.387207,180.0,29.579905,29.195265,178.387207,0.0
2,original,280333,4676307,9.624470,new,4.730880,204.424805,182.0,3.051872,2.724864,204.424805,0.0
3,subdivided,10304077,4676306,353.762409,old,380.517365,17077.544922,19270.0,352.820343,308.744202,17077.544922,0.0
4,subdivided,10304077,4676306,353.762409,ref,18.772991,178.387207,180.0,17.538561,16.248833,178.387207,0.0
5,subdivided,10304077,4676306,353.762409,new,21.387264,561.791992,408.0,18.203135,17.512800,561.791992,0.0
